In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from langchain.tools import tool
from langchain.agents import create_agent




In [10]:
loader = PyPDFLoader("../Data/medical_report.pdf")
docs = loader.load()
len(docs)


9

In [11]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
splitted_doc = splitter.split_documents(docs)
len(splitted_doc)

26

In [13]:
embedding = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2-preview"
)
vector_store = InMemoryVectorStore.from_documents(
    documents = splitted_doc,
    embedding=embedding,
    
)

In [14]:
same_record = vector_store.similarity_search("patient name")
same_record

[Document(id='14b9abcd-bd41-4185-b6e0-b91274163c46', metadata={'producer': 'PDFsharp 6.1.0', 'creator': 'Dr Lal PathLabs Limited', 'creationdate': '2025-07-11T09:10:48+00:00', 'author': 'Dr Lal PathLabs Limited', 'title': 'Patient Report', 'subject': 'Lal Pathlabs Report', 'keywords': 'Dr Lal PathLabs Limited', 'moddate': '2025-07-11T09:10:48+00:00', 'source': '../Data/medical_report.pdf', 'total_pages': 9, 'page': 0, 'page_label': '1'}, page_content='Report Status    \nFemale\n27 Years:\n:\n:\n:\nAge\nGender\nReported        \nP\n9/7/2025   4:56:00PM\nDR NITIN NAHAR\n474764803\nMs. NIKITA  CHUDHARY:\n:\n:\n:\n:\nName        \nLab No.    \nRef By \nCollected       \nA/c Status \n10/7/2025  6:31:50PM\nFinal\nCollected at            : Processed at             :BHOPAL CC-82\nMr Rachel V John Pata So Vitus John Mig 26 \nGraund,Indrapuri, Phone: 8770817968\n \nLPL - Bhopal Lab II\nPlot No.05, Mandakini Housing Society, Near \nApurti Shopping Mall, Kolar Main Road, \nBhopal, M.P. -  462042\n

In [ ]:
# agent = tools,llm,prompt

In [25]:
@tool
def retriever(query:str):
    """
        This tool can help you to reetrive the relevent data of Pdf Documents, and this pdf documents have details about medical report.
    """
    
    docs  = vector_store.similarity_search(query=query,k=4)
    context = ""
    for doc in docs:
        context = doc.page_content + "\n"
    return context


In [23]:
# retriver.invoke("Patient Name")
llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")
System_Prompt = """
    You are a helpful assistant that answers questions using retrieved context.
	ALWAYS use the `retriever` tool for questions requiring external knowledge.
"""


In [26]:


agent = create_agent(
    model=llm,
    tools=[retriever],
    system_prompt=System_Prompt
)

In [27]:
query = "What is the name of patient, and what is the name of Doctors"
response = agent.invoke({"messages":[{"role":"user", "content":query}]})

In [28]:
result = response["messages"][-1].content
print(result)

[{'type': 'text', 'text': 'Based on the medical report:\n\n* **Patient Name:** Ms. Nikita Choudhary\n* **Doctor Name (Ref By):** Dr. Nitin Nahar', 'extras': {'signature': 'El4KXAFpFH0TBUZpMAsxnwWmFJ1OMNeYPrYPGqfb0Yk2A2O9ox3Iq8Ms1OY2u24LUNBhb5EHGSb8NpfJevXIjP/JaEo/pc/qUNa9kh3nhFHcKQyz9ziQGAsM8i+X3ST/'}}]
